In [1]:
import os
import glob
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image, ImageFilter

In [9]:
# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    DATA_DIR = "/home/Joshua/Downloads/leopard_toad_identification/dataset/dataset_reid_crops"  # Folder containing cropped toad images
    IMG_SIZE = 1280                   # ResNet50 standard input size
    VAL_SPLIT = 0.2                  # 20% for validation
    SEED = 42

    BATCH_SIZE = 32                  # Lower this if you run out of GPU memory
    EPOCHS = 50
    LEARNING_RATE = 3e-4
    TEMPERATURE = 0.12                # Temperature for NT-Xent loss
    EMBEDDING_DIM = 128              # Dimension of the output vector
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_WORKERS = 4

In [3]:
# =============================================================================
# CUSTOM AUGMENTATIONS (FIXED ASPECT RATIO & CAMERA TRAP SIMULATION)
# =============================================================================
class ResizeAndPad:
    """
    Resizes the image to target_size while keeping aspect ratio, 
    then pads the shorter side with black (0) to make it a square.
    Prevents "squashing" of the leopard patterns.
    """
    def __init__(self, target_size, fill=0):
        self.target_size = target_size
        self.fill = fill

    def __call__(self, image):
        w, h = image.size
        scale = self.target_size / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        
        image = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
        
        delta_w, delta_h = self.target_size - new_w, self.target_size - new_h
        padding = (delta_w // 2, delta_h // 2, delta_w - (delta_w // 2), delta_h - (delta_h // 2))
        
        return T.functional.pad(image, padding, fill=self.fill)

class GaussianBlur:
    def __init__(self, sigma=(0.1, 2.0)):
        self.sigma = sigma

    def __call__(self, x):
        sigma = random.uniform(self.sigma[0], self.sigma[1])
        return x.filter(ImageFilter.GaussianBlur(radius=sigma))

class SimCLRTransform:
    def __init__(self, size):
        self.transform = T.Compose([
            # 1. Fix Aspect Ratio & Pad
            ResizeAndPad(size, fill=0),

            # 2. Safe Geometric Augmentations
            T.Pad(padding=int(size * 0.1)),
            T.RandomCrop(size=size),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=15),

            # 3. Photometric Augmentations (IR Simulation)
            T.RandomApply([T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            
            # 4. Blur (Simulating focus issues)
            T.RandomApply([GaussianBlur([0.1, 2.0])], p=0.5),

            # 5. Normalization
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)

In [4]:
# =============================================================================
# DATASET & MODEL
# =============================================================================
class ToadDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.files = file_paths
        self.transform = transform

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        image = Image.open(self.files[idx]).convert("RGB")
        return self.transform(image) if self.transform else image

class SimCLR(nn.Module):
    def __init__(self, base_model=models.resnet50, out_dim=128):
        super(SimCLR, self).__init__()
        self.backbone = base_model(weights=models.ResNet50_Weights.DEFAULT)
        dim_mlp = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.projection = nn.Sequential(
            nn.Linear(dim_mlp, 512),
            nn.ReLU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.projection(h)
        return h, z

In [6]:
# =============================================================================
# LOSS & UTILS
# =============================================================================
class NTXentLoss(nn.Module):
    def __init__(self, batch_size, temperature, device):
        super(NTXentLoss, self).__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.device = device
        self.criterion = nn.CrossEntropyLoss(reduction="sum")
        self.similarity_f = nn.CosineSimilarity(dim=2)
        self.mask = self._get_correlated_mask().to(device)

    def _get_correlated_mask(self):
        diag = np.eye(2 * self.batch_size)
        l1 = np.eye((2 * self.batch_size), k=self.batch_size)
        l2 = np.eye((2 * self.batch_size), k=-self.batch_size)
        mask = torch.from_numpy((diag + l1 + l2))
        return (1 - mask).type(torch.bool)

    def forward(self, z_i, z_j):
        z = torch.cat((z_i, z_j), dim=0)
        sim = self.similarity_f(z.unsqueeze(1), z.unsqueeze(0)) / self.temperature

        sim_i_j = torch.diag(sim, self.batch_size)
        sim_j_i = torch.diag(sim, -self.batch_size)
        
        positive_samples = torch.cat((sim_i_j, sim_j_i), dim=0).reshape(2 * self.batch_size, 1)
        negative_samples = sim[self.mask].reshape(2 * self.batch_size, -1)

        labels = torch.zeros(2 * self.batch_size).to(self.device).long()
        logits = torch.cat((positive_samples, negative_samples), dim=1)
        
        return self.criterion(logits, labels) / (2 * self.batch_size)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def plot_history(history, save_path="/home/Joshua/Downloads/leopard_toad_identification/identification/logs/training_trajectory.png"):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    plt.plot(epochs, history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
    plt.title('SimCLR Training Trajectory', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('NT-Xent Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

In [7]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    set_seed(Config.SEED)
    os.makedirs("/home/Joshua/Downloads/leopard_toad_identification/identification/weights", exist_ok=True)
    os.makedirs("/home/Joshua/Downloads/leopard_toad_identification/identification/logs", exist_ok=True)

    all_files = glob.glob(os.path.join(Config.DATA_DIR, "*.jpg"))
    if not all_files: raise ValueError(f"No images found in {Config.DATA_DIR}")

    train_files, val_files = train_test_split(all_files, test_size=Config.VAL_SPLIT, random_state=Config.SEED)
    print(f"Data Split: {len(train_files)} Train | {len(val_files)} Validation")

    train_transform = SimCLRTransform(Config.IMG_SIZE)
    val_transform = SimCLRTransform(Config.IMG_SIZE)

    train_loader = DataLoader(ToadDataset(train_files, transform=train_transform), batch_size=Config.BATCH_SIZE, shuffle=True, drop_last=True, num_workers=Config.NUM_WORKERS)
    val_loader = DataLoader(ToadDataset(val_files, transform=val_transform), batch_size=Config.BATCH_SIZE, shuffle=False, drop_last=True, num_workers=Config.NUM_WORKERS)

    model = SimCLR(base_model=models.resnet50, out_dim=Config.EMBEDDING_DIM).to(Config.DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)
    criterion = NTXentLoss(Config.BATCH_SIZE, Config.TEMPERATURE, Config.DEVICE)

    best_val_loss = float("inf")
    history = {'train_loss': [], 'val_loss':[]}

    print("Starting Training...")
    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0.0
        for (x_i, x_j) in train_loader:
            x_i, x_j = x_i.to(Config.DEVICE), x_j.to(Config.DEVICE)
            optimizer.zero_grad()
            _, z_i = model(x_i)
            _, z_j = model(x_j)
            loss = criterion(z_i, z_j)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (x_i, x_j) in val_loader:
                x_i, x_j = x_i.to(Config.DEVICE), x_j.to(Config.DEVICE)
                _, z_i = model(x_i)
                _, z_j = model(x_j)
                loss = criterion(z_i, z_j)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        scheduler.step()

        print(f"Epoch[{epoch+1}/{Config.EPOCHS}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict()}, "/home/Joshua/Downloads/leopard_toad_identification/identification/weights/best_simclr.pth")

    torch.save(model.backbone.state_dict(), "weights/resnet50_backbone_final.pth")
    with open("/home/Joshua/Downloads/leopard_toad_identification/identification/logs/training_history.json", "w") as f: json.dump(history, f)
    plot_history(history)
    print("\nTraining Complete. Backbone saved.")

In [10]:
main()

Data Split: 330 Train | 83 Validation
Starting Training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.56 GiB. GPU 0 has a total capacity of 47.39 GiB of which 1.57 GiB is free. Process 136245 has 378.00 MiB memory in use. Process 1247956 has 1.64 GiB memory in use. Including non-PyTorch memory, this process has 43.10 GiB memory in use. Of the allocated memory 42.76 GiB is allocated by PyTorch, and 30.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)